In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from src.config import load_config
from src.features.episodes import build_episodes
from src.models.anomaly_methods import (
    run_isolation_forest,
    run_distance_based_knn,
    run_density_based_lof,
    run_kernel_one_class_svm,
    compare_methods,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
%matplotlib inline

In [ ]:
config = load_config(str(PROJECT_ROOT / "config/config.yaml"))
gatherings = pd.read_parquet(PROCESSED_DIR / "gatherings_v1.parquet")
episodes = build_episodes(gatherings, config)

print(f"{len(episodes):,} episodes")
print(f"Columns: {episodes.columns.tolist()}")
episodes.head()

In [ ]:
FEATURES = ["n_slots", "median_size", "median_rssi", "n_gap_slots", "night_share"]
episodes[FEATURES].describe()

In [ ]:
ranked = run_isolation_forest(episodes, FEATURES, contamination=0.05)
print(f"Anomalies: {ranked['is_anomaly'].sum()} of {len(ranked)}")
ranked[FEATURES + ["anomaly_score", "is_anomaly"]].head(20)

In [ ]:
comparison = compare_methods(episodes, FEATURES, contamination=0.05)
comparison

In [ ]:
anomalies_dir = OUTPUTS_DIR / "anomalies"
anomalies_dir.mkdir(parents=True, exist_ok=True)

ranked.to_parquet(anomalies_dir / "episodes_anomalies_iforest.parquet")
comparison.to_csv(anomalies_dir / "method_comparison.csv")

print(f"✓ Saved to {anomalies_dir}/")

In [ ]:
print("="*80)
print("TOP 20 MOST ANOMALOUS EPISODES - DETAILED VIEW")
print("="*80)

top_anomalies = ranked.head(20).copy()
for idx, (rank, row) in enumerate(top_anomalies.iterrows(), 1):
    print(f"\nRank {rank}:")
    print(f"  Duration: {row['n_slots']} slots ({row['n_slots'] * 5} min = {row['n_slots'] / 12:.1f}h)")
    print(f"  Group size: {row['median_size']:.0f} people")
    print(f"  Signal strength: {row['median_rssi']:.1f} dBm")
    print(f"  Gaps: {row['n_gap_slots']} slots (fragmented)")
    print(f"  Nocturnal: {row['night_share']*100:.1f}% at night")
    print(f"  Anomaly score: {row['anomaly_score']:.4f}")

In [ ]:
# (Sample middle-range anomalies):
print("\n" + "="*80)
print("MIDDLE-RANGE ANOMALIES (ranks 500-520) - TRANSITIONAL CASES")
print("="*80)

mid_anomalies = ranked.iloc[499:520].copy()
print(mid_anomalies[FEATURES + ["anomaly_score", "is_anomaly"]])

In [ ]:
#(Examine normal episodes - tail):
print("\n" + "="*80)
print("NORMAL EPISODES (ranks 22,400+) - WHAT TYPICAL LOOKS LIKE")
print("="*80)

# The last 200 are least anomalous (by score)
normal_episodes = ranked.tail(200)
print(f"\nStatistics for least anomalous 200 episodes:")
print(normal_episodes[FEATURES].describe().round(2))

print(f"\nSample of normal episodes (ranks 22,300-22,330):")
print(normal_episodes.iloc[:30][FEATURES + ["anomaly_score", "is_anomaly"]])

In [ ]:
 #(Comparison: anomalies vs normal):
print("\n" + "="*80)
print("ANOMALIES vs NORMAL - STATISTICAL COMPARISON")
print("="*80)

anomalies_only = ranked[ranked['is_anomaly']]
normal_only = ranked[~ranked['is_anomaly']]

comparison_df = pd.DataFrame({
    'Anomalies (1127)': anomalies_only[FEATURES].describe().loc[['mean', 'std', 'min', 'max']],
    'Normal (21403)': normal_only[FEATURES].describe().loc[['mean', 'std', 'min', 'max']]
})

print(comparison_df.round(2))

In [ ]:
#Filtering strategies & recommendations):
print("\n" + "="*80)
print("FILTERING STRATEGIES")
print("="*80)

print("\n1. REMOVE ALL ANOMALIES")
print(f"   Keeps: {len(normal_only):,} episodes")
print(f"   Removes: {len(anomalies_only):,} episodes (5%)")
print(f"   Best for: Conservative analysis (exclude potential glitches)")

print("\n2. SEPARATE ANOMALIES (keep both, tag them)")
print(f"   Keeps: {len(ranked):,} episodes total")
print(f"   Tags: 'is_anomaly' column = True/False")
print(f"   Best for: Exploratory analysis (preserve data, mark outliers)")

print("\n3. THRESHOLD-BASED (more aggressive)")
# Flag top 1% instead of 5%
top_1pct = int(len(ranked) * 0.01)
aggressive_threshold = ranked.iloc[top_1pct - 1]['anomaly_score']
print(f"   Removes: {top_1pct:,} episodes (top 1%)")
print(f"   Threshold: anomaly_score > {aggressive_threshold:.4f}")
print(f"   Best for: Strict quality control")

print("\n4. THRESHOLD-BASED (more conservative)")
# Flag top 10% instead of 5%
top_10pct = int(len(ranked) * 0.10)
conservative_threshold = ranked.iloc[top_10pct - 1]['anomaly_score']
print(f"   Removes: {top_10pct:,} episodes (top 10%)")
print(f"   Threshold: anomaly_score > {conservative_threshold:.4f}")
print(f"   Best for: Exploratory work (maximize data retention)")

print("\n" + "="*80)
print("RECOMMENDATION")
print("="*80)
print("Start with SEPARATE ANOMALIES (strategy 2):")
print("  ✓ Preserves full dataset")
print("  ✓ Allows post-hoc removal if needed")
print("  ✓ Enables investigation of what makes them anomalous")

In [ ]:
# Save both versions
anomalies_dir = OUTPUTS_DIR / "anomalies"

# Version 1: All episodes with anomaly flags
ranked.to_parquet(anomalies_dir / "episodes_anomalies_iforest.parquet")

# Version 2: Only normal episodes
normal_only.to_parquet(anomalies_dir / "episodes_normal_only.parquet")

# Version 3: Only anomalies
anomalies_only.to_parquet(anomalies_dir / "episodes_anomalies_only.parquet")

print(f"✓ Saved 3 versions:")
print(f"  1. episodes_anomalies_iforest.parquet       ({len(ranked):,} rows, all with flags)")
print(f"  2. episodes_normal_only.parquet             ({len(normal_only):,} rows, clean)")
print(f"  3. episodes_anomalies_only.parquet          ({len(anomalies_only):,} rows, outliers only)")

In [ ]:
anomalies_dir = OUTPUTS_DIR / "anomalies"

# Load the ranked results (from previous cell)
# ranked should already exist, but if not:
# ranked = pd.read_parquet(anomalies_dir / "episodes_anomalies_iforest.parquet")

normal_only = ranked[~ranked['is_anomaly']]
anomalies_only = ranked[ranked['is_anomaly']]

# Version 2: Only normal episodes
normal_only.to_parquet(anomalies_dir / "episodes_normal_only.parquet")

# Version 3: Only anomalies
anomalies_only.to_parquet(anomalies_dir / "episodes_anomalies_only.parquet")

print(f"✓ Saved 3 versions:")
print(f"  1. episodes_anomalies_iforest.parquet       ({len(ranked):,} rows, all with flags)")
print(f"  2. episodes_normal_only.parquet             ({len(normal_only):,} rows, clean)")
print(f"  3. episodes_anomalies_only.parquet          ({len(anomalies_only):,} rows, outliers only)")